# Manipulation maps — what an attacker is allowed to change

Every attack in this repository is constrained by these arrays. Each Drebin feature is categorised by type (permission, activity, API call, URL, …), and from the category comes the rule: some features may only be **inserted**, some only **removed**, some not touched at all — because an adversarial example still has to be a working app. The outputs are the `insertion_array` and `removal_array` used by the attacks and by the clustering study in `02-detectors/`.

*Consolidated from `get_manipulation.ipynb` and `get_manipulation2.ipynb` (originally in `mal_adv3`). The second notebook is the developed version; the categorisation function it calls is defined in the first, so the two are merged here in dependency order. Outputs cleared.*

## Step 1 — Load the Drebin vocabulary

In [ ]:
import pickle
import collections
import re
import numpy as np
import gdown

In [ ]:
download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/vocabulary.pkl',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/vocabulary_info.pkl',
]


output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
# Open the .pkl file
with open('/content/vocabulary.pkl', 'rb') as f:
    # Load the object
    vocabulary = pickle.load(f)

# Close the file
f.close()

print(len(vocabulary))

# Open the .pkl file
with open('/content/vocabulary_info.pkl', 'rb') as f:
    # Load the object
    info = pickle.load(f)

# Close the file
f.close()

print(len(info))

In [ ]:
vocabulary[:5]

In [ ]:
# Remove "b" from the beginning of each element
vocab = [element.decode('utf-8') for element in vocabulary]
vocab[:5]

In [ ]:
info

In [ ]:
# Remove "b" from the beginning of each key
vocab_info = {key.decode('utf-8'): value for key, value in info.items()}

# Print the modified dictionary
print(vocab_info)

## Step 2 — Categorise every feature by type

In [ ]:
COMP = {
    "Permission": "permission",
    "Activity": "activity",
    "Service": "service",
    "Receiver": "receiver",
    "Provider": "provider",
    "Hardware": "hardware",
    "Intentfilter": 'intent-filter',
    "Android_API": "android_api",
    "Java_API": "java_api",
    "User_String": "const-string",
    "User_Class": "user_class",
    "User_Method": "user_method",
    "OpCode": "opcode",
    "Asset": "asset",
    "Notdefined": 'not_defined'
}


def get_word_category(vocabulary, vocabulary_info, defined_comp):
    """
    Get the category for each word in vocabulary, based on the COMP in conf file
    :rtype: object
    """

    def _api_check(dalvik_code_line_list):
        for code_line in dalvik_code_line_list:
            invoke_match = re.search(
                r'(?P<invokeType>invoke\-([^ ]*?)) (?P<invokeParam>([vp0-9,. ]*?)), (?P<invokeObject>L(.*?);|\[L(.*?);)->(?P<invokeMethod>(.*?))\((?P<invokeArgument>(.*?))\)(?P<invokeReturn>(.*?))$',
                code_line)
            if invoke_match is None:
                return defined_comp['Notdefined']
            if invoke_match.group('invokeType') == 'invoke-virtual' or invoke_match.group(
                    'invokeType') == 'invoke-virtual/range' or \
                    invoke_match.group('invokeType') == 'invoke-static' or \
                    invoke_match.group('invokeType') == 'invoke-static/range':
                if invoke_match.group('invokeObject').startswith('Landroid'):
                    return defined_comp['Android_API']
                elif invoke_match.group('invokeObject').startswith('Ljava'):
                    return defined_comp['Java_API']
                else:
                    return defined_comp['Notdefined']
            else:
                return defined_comp['Notdefined']

    word_cat_dict = collections.defaultdict()
    for w in vocabulary:
        if 'ActivityList_' in w:
            word_cat_dict[w] = defined_comp['Activity']
        elif 'RequestedPermissionList_' in w:
            word_cat_dict[w] = defined_comp['Permission']
        elif 'ServiceList_' in w:
            word_cat_dict[w] = defined_comp['Service']
        elif 'ContentProviderList_' in w:
            word_cat_dict[w] = defined_comp['Provider']
        elif 'BroadcastReceiverList_' in w:
            word_cat_dict[w] = defined_comp['Receiver']
        elif 'HardwareComponentsList_' in w:
            word_cat_dict[w] = defined_comp['Hardware']
        elif 'IntentFilterList_' in w:
            word_cat_dict[w] = defined_comp['Intentfilter']
        elif 'UsedPermissionsList_' in w:
            word_cat_dict[w] = defined_comp['Notdefined']
        elif 'RestrictedApiList_' in w:
            word_cat_dict[w] = _api_check(vocabulary_info[w])
        elif 'SuspiciousApiList' in w:
            word_cat_dict[w] = _api_check(vocabulary_info[w])
        elif 'URLDomainList' in w:
            word_cat_dict[w] = defined_comp['User_String']
        else:
            word_cat_dict[w] = defined_comp['Notdefined']
    return word_cat_dict

In [ ]:
def get_word_category(vocabulary, vocabulary_info, defined_comp):
    """
    Get the category for each word in vocabulary, based on the COMP in conf file
    :rtype: object
    """

    word_cat_dict = {}
    for w in vocabulary:
        if 'ActivityList_' in w:
            word_cat_dict[w] = defined_comp['Activity']
        elif 'RequestedPermissionList_' in w:
            word_cat_dict[w] = defined_comp['Permission']
        elif 'ServiceList_' in w:
            word_cat_dict[w] = defined_comp['Service']
        elif 'ContentProviderList_' in w:
            word_cat_dict[w] = defined_comp['Provider']
        elif 'BroadcastReceiverList_' in w:
            word_cat_dict[w] = defined_comp['Receiver']
        elif 'HardwareComponentsList_' in w:
            word_cat_dict[w] = defined_comp['Hardware']
        elif 'IntentFilterList_' in w:
            word_cat_dict[w] = defined_comp['Intentfilter']
        elif 'UsedPermissionsList_' in w:
            word_cat_dict[w] = defined_comp['UsedPermissions']
        elif 'RestrictedApiList_' in w:
            word_cat_dict[w] = defined_comp['SYS_API']
        elif 'SuspiciousApiList' in w:
            word_cat_dict[w] = defined_comp['SYS_API']
        elif 'URLDomainList' in w:
            word_cat_dict[w] = defined_comp['User_String']
        else:
            word_cat_dict[w] = defined_comp['Notdefined']
            print('Notdefined')
    return word_cat_dict

In [ ]:
vocab_type_dic = get_word_category(vocab, vocab_info, COMP)
vocab_type_dic

In [ ]:
# Get the values as a list
vocab_type = list(vocab_type_dic.values())
vocab_type[:5]

## Step 3 — The manipulation rule per feature

In [ ]:
def get_manipulation(vocab,vocab_info,vocab_type):
        """
        All features are insertable and the apis that have public descriptor can be hidden by java reflection.
        For efficiency and simplicity consideration, this function only returns a mask to filter out the apis that are non-refelectable.
        This means the value "1" in the mask vector corresponds to a removable feature, and "0" means otherwise.
        """
        manipulation = np.zeros((len(vocab),), dtype=np.float32)
        for i, v, v_type in zip(range(len(vocab)), vocab, vocab_type):
            if v_type in ['activity', 'service', 'receiver', 'provider']:
                manipulation[i] = 1.
            if v_type == 'User_String':
                manipulation[i] = 1.
            if v_type == 'api' and approx_check_public_method(v, vocab_info.get(v)):
                #print(v)
                #print(vocab_info.get(v))
                manipulation[i] = 1.
        return manipulation

def approx_check_public_method(word, word_info):
    assert isinstance(word, str) and isinstance(word_info, set)
    # see: https://docs.oracle.com/javase/specs/jvms/se10/html/jvms-2.html#jvms-2.12, we do not hide reflection-related API again
    if re.search(r'\<init\>|\<clinit\>', word) is None and \
            re.search(
                r'Ljava\/lang\/reflect\/Method;->invoke|Ljava\/lang\/Object;->getClass|Ljava\/lang\/Class;->getMethod',
                word) is None and \
            all(
                [re.search(r'invoke\-virtual|invoke\-static|invoke\-interface', info) for info in word_info]):
        return True

In [ ]:
manipulation = get_manipulation(vocab,vocab_info,vocab_type)
print(sum(manipulation))

In [ ]:
manipulation[:10]

## Step 4 — Build and save the insertion / removal arrays

In [ ]:
OPERATOR = {
    # insert
    0: "insert",
    # remove
    1: "remove"
}

INSTR_ALLOWED = {
    OPERATOR[0]: [COMP['Permission'],
                  COMP['Activity'],
                  COMP['Service'],
                  COMP['Receiver'],
                  COMP['Hardware'],
                  COMP['Intentfilter'],
                  COMP['Android_API'],
                  COMP['User_String']
                  ],
    OPERATOR[1]: [COMP['Activity'],
                  COMP['Service'],
                  COMP['Receiver'],
                  COMP['Provider'],
                  COMP['Android_API'],
                  COMP['User_String']
                  ]
}



def get_mod_array(vocab, word_catagory_dict):
    """
    get binary indicator of showing the feature can be either modified or not
    '1' means modifiable and '0' means not
    """
    insertion_array = []
    removal_array = []

    insertion_array = np.zeros(len(vocab), )
    removal_array = np.zeros(len(vocab), )

    for i, word in enumerate(vocab):
        cat = word_catagory_dict.get(word)
        if cat is not None:
            if cat in INSTR_ALLOWED[OPERATOR[0]]:
                insertion_array[i] = 1
            else:
                insertion_array[i] = 0
            if cat in INSTR_ALLOWED[OPERATOR[1]]:
                removal_array[i] = 1
            else:
                removal_array[i] = 0
        else:
            raise ValueError("Incompatible value.")

    return insertion_array, removal_array

In [ ]:
insertion_array, removal_array = get_mod_array(vocab, vocab_type_dic)

print(sum(insertion_array))
print(sum(removal_array))

In [ ]:
# File path to save the numbers
file_path = 'insertion_array.pkl'

# Open the file in binary write mode
with open(file_path, 'wb') as file:
    # Use pickle to dump the list of numbers
    pickle.dump(insertion_array, file)

print("Numbers saved to:", file_path)


# File path to save the numbers
file_path = 'removal_array.pkl'

# Open the file in binary write mode
with open(file_path, 'wb') as file:
    # Use pickle to dump the list of numbers
    pickle.dump(removal_array, file)

print("Numbers saved to:", file_path)

## Step 5 — Check the counts per category

In [ ]:
from collections import Counter

# Count the occurrences of each element
element_counts = Counter(vocab_type)

# Print the counts
for element, count in element_counts.items():
    print(f"{element}: {count}")

In [ ]:
# Specific value to search for
specific_value = 'not_defined'

# Initialize a list to store keys with the specific value
keys_with_specific_value = []

# Iterate over the items of the dictionary
for key, value in vocab_type_dic.items():
    # Check if the value matches the specific value
    if value == specific_value:
        # Add the key to the list
        keys_with_specific_value.append(key)

# Print the keys with the specific value
print(keys_with_specific_value)

In [ ]:
i = 0
j = 0
for v, v_type in zip(vocab, vocab_type):
  if v_type == 'api' :
    i+=1
  if v_type == 'api' and approx_check_public_method(v, vocab_info.get(v)):
    j+=1

print(i)
print(j)